# Bab 26: Analisis Wavelet

Notebook pendamping buku *Data Mining dengan Python: Konsep, Algoritme, dan Studi Kasus dengan CRISP-DM*. Setiap sel kode sama dengan listing di buku; keluaran yang dicetak di buku ada di berkas `.out` di folder `kode/bab26/`.

Jalankan sel secara berurutan.

In [ ]:
# Jalankan sel ini sekali di awal sesi.
import os, sys, subprocess
if "google.colab" in sys.modules and not os.path.exists("kode"):
    if not os.path.exists("data-mining-dengan-python"):
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/bsnegara/data-mining-dengan-python"])
    os.chdir("data-mining-dengan-python")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "-r", "kode/requirements-colab.txt"])
while not os.path.exists("kode/data") and os.getcwd() != "/":
    os.chdir("..")                     # cari akar repositori
sys.path.insert(0, "kode/bab26")
print("Folder kerja:", os.getcwd())

## Listing 26.1: Transformasi Haar dengan tangan dan dengan PyWavelets

`kode/bab26/haar_mini.py`

In [ ]:
import numpy as np
import pywt

x = np.array([0, 2, 0, 2, 6, 8, 6, 8], float)
a, rinci = x.copy(), []
while len(a) > 1:
    rata = (a[0::2] + a[1::2]) / 2
    beda = (a[0::2] - a[1::2]) / 2
    rinci.append(beda)
    print(f"tingkat {len(rinci)}: rata-rata {rata}, rincian {beda}")
    a = rata
print("koefisien Haar:", a, *rinci[::-1])

def balik(a, rinci):
    for d in rinci[::-1]:
        y = np.empty(2 * len(a))
        y[0::2], y[1::2] = a + d, a - d
        a = y
    return a

print("rekonstruksi  :", balik(a, rinci))
nol = [np.zeros_like(rinci[0])] + rinci[1:]
print("tanpa rincian tingkat 1:", balik(a, nol))

c = pywt.wavedec(x, "haar")
print("\nPyWavelets (ortonormal):")
for nama, v in zip(["cA3", "cD3", "cD2", "cD1"], c):
    print(f"  {nama}: {np.round(v, 3)}")
skala = [2 ** 1.5, 2 ** 1.5, 2, np.sqrt(2)]
print("dibagi 2^(j/2):", [np.round(v / s, 3).tolist()
                         for v, s in zip(c, skala)])
print("energi sama:", np.sum(x ** 2),
      round(sum((v ** 2).sum() for v in c), 6))

## Listing 26.2: Fungsi penghalusan dan RoR

`kode/bab26/halus.py`

In [ ]:
import numpy as np
import pandas as pd
import pywt


def halus_wavelet(x, wavelet="sym8", tingkat=5, mode="soft"):
    """Penghilangan derau: ambang universal pada koefisien rincian."""
    c = pywt.wavedec(x, wavelet, level=tingkat, mode="smooth")
    sigma = np.median(np.abs(c[-1])) / 0.6745          # MAD tingkat 1
    ambang = sigma * np.sqrt(2 * np.log(len(x)))
    c = [c[0]] + [pywt.threshold(d, ambang, mode) for d in c[1:]]
    return pywt.waverec(c, wavelet, mode="smooth")[:len(x)]


def halus_wavelet_ti(x, geser=16, **opsi):
    """Invarian geser (cycle spinning): rata-rata hasil dari sinyal
    yang dipotong k titik di depan, k = 0, ..., geser-1."""
    jumlah, n = np.zeros(len(x)), np.zeros(len(x))
    for k in range(geser):
        jumlah[k:] += halus_wavelet(x[k:], **opsi)
        n[k:] += 1
    return jumlah / n


def halus_median(x, w=9):
    return pd.Series(x).rolling(w, center=True,
                                min_periods=1).median().to_numpy()


def halus_rata(x, w=15):
    return pd.Series(x).rolling(w, center=True,
                                min_periods=1).mean().to_numpy()


def ror(x, w=30):
    """Laju kenaikan suhu (derajat per menit) dari selisih w detik."""
    return pd.Series(x).diff(w).to_numpy() * 60 / w

## Listing 26.3: Eksperimen penghilangan derau

`kode/bab26/hilang_derau.py`

In [ ]:
import sys
from functools import partial
import numpy as np
import pywt
from halus import (halus_median, halus_rata, halus_wavelet,
                   halus_wavelet_ti, ror)
sys.path.insert(0, "kode/data")
from bangkitkan_data_lanjut import kurva_sangrai

def crack(x, t0=150, suhu=196):
    """Detik pertama suhu biji melewati 196 C setelah titik balik."""
    return int(np.argmax(x[t0:] > suhu)) + t0


rng = np.random.default_rng(2627)
metode = {"tanpa penghalusan": lambda x: x,
          "rata-rata bergerak 15": halus_rata,
          "median bergerak 9": halus_median,
          "wavelet sym8 lunak": halus_wavelet,
          "wavelet haar lunak": partial(halus_wavelet,
                                        wavelet="haar"),
          "wavelet sym8 keras": partial(halus_wavelet, mode="hard"),
          "wavelet keras + geser": partial(halus_wavelet_ti,
                                           mode="hard")}
galat = {m: [] for m in metode}
for _ in range(40):
    t, benar, _ = kurva_sangrai(rng, int(rng.normal(690, 25)),
                                rng.normal(210, 2.5))
    x = benar + rng.normal(0, 0.7, len(t))
    rb = ror(benar, 10)
    for m, f in metode.items():
        h = f(x)
        balik = np.argmin(h[:200]) - np.argmin(benar[:200])
        galat[m].append([np.sqrt(np.mean((h - benar) ** 2)),
                         np.sqrt(np.nanmean((ror(h, 10) - rb) ** 2)),
                         abs(balik), crack(h) - crack(benar)])
print(f"{'metode':22s}   RMSE   RMSE  |galat|   galat")
print(f"{'':22s}   suhu    RoR    balik   crack")
for m, g in galat.items():
    g = np.array(g).mean(0)
    print(f"{m:22s} {g[0]:6.3f} {g[1]:6.2f} {g[2]:7.1f}s"
          f" {g[3]:6.1f}s")

c = pywt.wavedec(x, "sym8", level=5, mode="smooth")
sigma = np.median(np.abs(c[-1])) / 0.6745
ambang = sigma * np.sqrt(2 * np.log(len(x)))
nol = sum((np.abs(d) <= ambang).sum() for d in c[1:])
print(f"\nsigma derau dari rincian tingkat 1: {sigma:.3f}"
      " (benar 0,7)")
print(f"ambang universal {ambang:.2f}; koefisien rincian dinolkan: "
      f"{nol} dari {sum(len(d) for d in c[1:])}")
print("panjang koefisien per tingkat:", [len(v) for v in c])

## Listing 26.4: Kurva sangrai bersih dan metrik pascabatch

`kode/bab26/studi_kasus.py`

In [ ]:
import numpy as np
import pandas as pd
from halus import halus_median, halus_wavelet_ti, ror

s = pd.read_csv("kode/data/sensor_sangrai.csv")
b = pd.read_csv("kode/data/batch_sangrai.csv").set_index("id_batch")
kurva = {k: g.suhu_biji.to_numpy() for k, g in s.groupby("id_batch")}


def lonjakan(x, k=10):
    """Rincian Haar tingkat 1 tak tercuplik: (x_t - x_t-1)/akar 2."""
    d = np.diff(x) / np.sqrt(2)
    sigma = np.median(np.abs(d)) / 0.6745
    tepi = np.where(np.abs(d) > k * sigma)[0]
    episode = []                  # tepi yang berjarak <= 3 detik
    for t in tepi:                # dianggap satu episode lonjakan
        if episode and t - episode[-1][-1] <= 3:
            episode[-1].append(t)
        else:
            episode.append([t])
    titik = [i for e in episode for i in range(e[0] + 1, e[-1] + 1)]
    return titik, len(episode), sigma


hasil, bersih = [], {}
for k, x in kurva.items():
    idx, n_lonjak, sigma = lonjakan(x)
    y = x.astype(float).copy()
    y[idx] = np.nan
    y = pd.Series(y).interpolate().to_numpy()
    z = halus_wavelet_ti(y, mode="hard")
    bersih[k] = z
    balik = int(np.argmin(z[:200]))
    lewat = z[150:] > 196
    crack = int(np.argmax(lewat)) + 150 if lewat.any() else np.nan
    hasil.append({"batch": k, "lonjak": n_lonjak,
                  "sigma": round(sigma, 2), "balik": balik,
                  "suhu_balik": round(z[balik], 1), "crack": crack,
                  "crack_log": b.detik_crack[k],
                  "DTR": (len(x) - 1 - crack) / (len(x) - 1),
                  "catatan": b.catatan_operator[k]})
h = pd.DataFrame(hasil).set_index("batch")
h["crack"] = h.crack.astype("Int64")
print("sigma derau sensor: median", h.sigma.median(),
      " maksimum", h.sigma.max(), f"({h.sigma.idxmax()})")
print("lonjakan terdeteksi:", h.lonjak[h.lonjak > 0].to_dict())

print("\nmetrik pascabatch (kurva bersih):")
pilih = ["B001", "B002", "B003", "B017", "B038", "B055"]
print(h.loc[pilih, ["balik", "suhu_balik", "crack", "crack_log",
                    "DTR", "catatan"]].round(3).to_string())
selisih = (h.crack - h.crack_log).dropna()
jauh = list(selisih.index[selisih.abs() > 30])
print(f"crack bersih - crack log: median {selisih.median():.0f} "
      f"detik; beda > 30 detik: {jauh}")
normal = h[h.catatan == "normal"]
print(f"DTR batch normal: {normal.DTR.mean():.3f} +- "
      f"{normal.DTR.std():.3f}")

acuan = [k for k in b.index[:30] if h.catatan[k] == "normal"
         and h.lonjak[k] == 0]
for nama, f in [("median 9 + selisih 30 s (Bab 22)",
                 lambda k: ror(halus_median(kurva[k]), 30)),
                ("wavelet + selisih 10 s",
                 lambda k: ror(bersih[k], 10))]:
    R = np.array([f(k)[200:480] for k in acuan])
    mu, sd = R.mean(0), R.std(0)
    luar = np.abs((f("B038")[200:480] - mu) / sd) > 3
    run = np.convolve(luar, np.ones(5), "valid") == 5
    print(f"{nama}: B038 bersinyal pada detik "
          f"{200 + np.argmax(run) + 4}")
pd.DataFrame({"t": np.arange(200, 480),
              "med": ror(halus_median(kurva["B038"]), 30)[200:480],
              "wav": ror(bersih["B038"], 10)[200:480],
              "mu": mu, "sd": sd}).to_csv(
    "gambar/bab26/ror.dat", sep=" ", index=False, float_format="%.3f")